In [8]:
import pandas as pd
import numpy as np

# 1. Load the raw dataset
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"Initial Shape: {df.shape}")

# 2. Fix Data Anomalies: 'TotalCharges' is stored as string due to blank spaces (" ") for new customers (tenure = 0)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Impute missing TotalCharges with 0.0 (since tenure is 0, they haven't been billed yet)
missing_count = df['TotalCharges'].isnull().sum()
print(f"Missing TotalCharges fixed: {missing_count}")
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

# 3. Standardize Binary & Categorical Columns for Clean SQL/Power BI Modeling
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
df['Churn_Flag'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Clean redundant 'No internet service' / 'No phone service' labels for cleaner grouping
replace_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]
for col in replace_cols:
    df[col] = df[col].replace({'No internet service': 'No'})

df['MultipleLines'] = df['MultipleLines'].replace({'No phone service': 'No'})

# 4. Feature Engineering: Create Business Segments for Retention Analysis
# A. Tenure Cohorts
def assign_tenure_cohort(tenure):
    if tenure <= 12:
        return '0-12 Months'
    elif tenure <= 24:
        return '13-24 Months'
    elif tenure <= 48:
        return '25-48 Months'
    else:
        return '49+ Months'

df['Tenure_Cohort'] = df['tenure'].apply(assign_tenure_cohort)

# B. Monthly Revenue Tiers
df['Revenue_Tier'] = pd.cut(
    df['MonthlyCharges'],
    bins=[0, 35, 70, 120],
    labels=['Low (<$35)', 'Medium ($35-$70)', 'High (>$70)']
)

# C. Total Active Add-On Services Count (measures customer stickiness)
addon_cols = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
df['Active_Addons_Count'] = (df[addon_cols] == 'Yes').sum(axis=1)

# 5. Validate & Export Cleaned Dataset
print("Null values remaining:", df.isnull().sum().sum())
print(df[['customerID', 'tenure', 'Tenure_Cohort', 'MonthlyCharges', 'Revenue_Tier', 'Active_Addons_Count', 'Churn']].head())

df.to_csv('../data/cleaned_telco_churn.csv', index=False)
print("Successfully exported cleaned_telco_churn.csv!")

Initial Shape: (7043, 21)
Missing TotalCharges fixed: 11
Null values remaining: 0
   customerID  tenure Tenure_Cohort  MonthlyCharges      Revenue_Tier  \
0  7590-VHVEG       1   0-12 Months           29.85        Low (<$35)   
1  5575-GNVDE      34  25-48 Months           56.95  Medium ($35-$70)   
2  3668-QPYBK       2   0-12 Months           53.85  Medium ($35-$70)   
3  7795-CFOCW      45  25-48 Months           42.30  Medium ($35-$70)   
4  9237-HQITU       2   0-12 Months           70.70       High (>$70)   

   Active_Addons_Count Churn  
0                    1    No  
1                    3    No  
2                    3   Yes  
3                    3    No  
4                    1   Yes  
Successfully exported cleaned_telco_churn.csv!
